# 1. Prepare all Nencki-Symfonia oddball subjects
Enable Internet; CPU is sufficient. Run all 42 subjects, save the output version, then attach it to notebook 02. Raw data is downloaded one subject at a time and cleaned up. Signal-quality exclusions are logged, never hidden.

In [ ]:
from pathlib import Path
import subprocess, sys, shutil

REPO = Path('/kaggle/working/TTKHTKUD_26-27')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/trungnd06/TTKHTKUD_26-27.git', str(REPO)], check=True)
SOURCE = REPO / 'kaggle'
assert (SOURCE / 'prepare_data.py').is_file(), 'This clone does not contain kaggle/. Use the updated repository.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE / 'requirements.txt')], check=True)
print('Code:', SOURCE)


In [ ]:
# Optional: an attached BIDS raw dataset root. None = download from public OpenNeuro.
RAW_ROOT = None  # Example: '/kaggle/input/my-nencki-raw/ds004621'
# Optional: previously saved processed folder, to resume completed subjects.
RESUME_PROCESSED = None
OUTPUT = Path('/kaggle/working/nencki_full/processed')
SUBJECTS = list(range(1, 43))  # Keep all 42 for the full experiment.

if RESUME_PROCESSED is not None:
    source = Path(RESUME_PROCESSED).resolve()
    assert source.is_dir() and source != OUTPUT.resolve()
    shutil.copytree(source, OUTPUT, dirs_exist_ok=True)
command = [sys.executable, str(SOURCE / 'prepare_data.py'), '--output', str(OUTPUT),
           '--subjects', *map(str, SUBJECTS)]
if RAW_ROOT is not None:
    command += ['--raw-root', str(RAW_ROOT)]
subprocess.run(command, check=True)


In [ ]:
import json
summary = json.loads((OUTPUT / 'preparation_summary.json').read_text())
print('All 42 attempted:', summary['all_42_attempted'])
print('Retained subjects:', len(summary['included_subjects']))
print('Quality exclusions:', summary['excluded_subjects'])
print('Standard / target:', summary['n_standard'], summary['n_target'])
print('Shape:', summary['shape'])
print('Output MiB:', round(sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file()) / 1024**2, 1))


## Save the preparation output
Choose **Save Version** and retain output files. In notebook 02, Add Input and attach this output. Alternatively download the whole processed folder and create your own Kaggle Dataset. Keep `preparation_summary.json` beside `epochs_full.npz`. The script can resume subjects only from files actually retained or restored in a later session.